# S02 — From Linear Models to MLPs

**Week 1 · Wed Aug 26, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s02_from_linear_models_to_mlps.ipynb)

Every cell below is a worked example from the [S02 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s02/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s02.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s02.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch  # noqa: F401
print("environment ready")

## The perceptron and its learning rule


*Expected output starts with:* `separable data: mistakes per epoch = [2, 0]`


In [ ]:
import numpy as np

np.random.seed(0)

def train_perceptron(X, y, epochs=50):
    """Rosenblatt's rule: on each mistake, add y_i * x_i to the weights.

    y must be in {-1, +1}. Returns mistakes made in each epoch.
    """
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            if yi * (w @ xi + b) <= 0:      # wrong side (or on) the boundary
                w += yi * xi
                b += yi
                mistakes += 1
        history.append(mistakes)
        if mistakes == 0:
            break
    return w, b, history

# Dataset 1: linearly separable — two blobs with a gap between them
X_sep = np.vstack([np.random.randn(50, 2) + [2, 2],
                   np.random.randn(50, 2) + [-2, -2]])
y_sep = np.array([1] * 50 + [-1] * 50)

# Dataset 2: XOR — four points no line can split correctly
X_xor = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1]], dtype=float)
y_xor = np.array([-1, -1, 1, 1])

w, b, hist = train_perceptron(X_sep, y_sep)
print(f"separable data: mistakes per epoch = {hist}")
print(f"separable data: converged after {len(hist)} epochs, w = {w}, b = {b}")

w, b, hist = train_perceptron(X_xor, y_xor, epochs=1000)
print(f"XOR data: ran {len(hist)} epochs, mistakes in the last 5 epochs = {hist[-5:]}")

## Why stacking linear layers is not enough


*Expected output starts with:* `10-layer output shape: (100, 3), single-layer W_eff shape: (4, 3)`


In [ ]:
import numpy as np

np.random.seed(0)

# Ten linear layers, no activations, random weights
sizes = [4, 16, 16, 16, 16, 16, 16, 16, 16, 16, 3]
Ws = [0.5 * np.random.randn(a, b) for a, b in zip(sizes[:-1], sizes[1:])]
bs = [0.1 * np.random.randn(b) for b in sizes[1:]]

def deep_linear(X):
    h = X
    for W, b in zip(Ws, bs):
        h = h @ W + b
    return h

# Collapse the whole stack into a single (W_eff, b_eff) by composing the maps
W_eff = np.eye(sizes[0])
b_eff = np.zeros(sizes[0])
for W, b in zip(Ws, bs):
    W_eff = W_eff @ W
    b_eff = b_eff @ W + b

X = np.random.randn(100, 4)
out_deep = deep_linear(X)
out_flat = X @ W_eff + b_eff
print(f"10-layer output shape: {out_deep.shape}, single-layer W_eff shape: {W_eff.shape}")
print(f"max |difference| between 10 layers and 1 layer: {np.max(np.abs(out_deep - out_flat)):.2e}")

total_params = sum(W.size + b.size for W, b in zip(Ws, bs))
print(f"parameters in the stack: {total_params}, in the collapsed layer: {W_eff.size + b_eff.size}")

## Activation functions: sigmoid, tanh, ReLU


*Expected output starts with:* `z      d_sigmoid  d_tanh   d_relu`


In [ ]:
import numpy as np

np.random.seed(0)

def sigmoid(z): return 1 / (1 + np.exp(-z))

def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)

def d_tanh(z): return 1 - np.tanh(z) ** 2

def d_relu(z): return np.asarray(z > 0, dtype=float)

print("z      d_sigmoid  d_tanh   d_relu")
for z in [0.0, 2.0, 5.0, 10.0]:
    print(f"{z:>4.1f}   {d_sigmoid(z):8.4f}  {d_tanh(z):7.4f}  {d_relu(z):6.1f}")

# What saturation does to a deep chain of gradients: multiply 20 layer-local
# derivatives together, using typical pre-activation values ~N(0, 1.5^2)
z = 1.5 * np.random.randn(20)
print(f"\nproduct of 20 sigmoid derivatives: {np.prod(d_sigmoid(z)):.2e}")
print(f"product of 20 tanh derivatives:    {np.prod(d_tanh(z)):.2e}")
print(f"product of 20 relu derivatives:    {np.prod(d_relu(z)):.2e}")

## Watching gradients vanish in a real network


*Expected output starts with:* `gradient norm of the weight matrix at each depth (12 linear layers):`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_net(act, depth=12, width=64):
    layers = [nn.Linear(2, width), act()]
    for _ in range(depth - 2):
        layers += [nn.Linear(width, width), act()]
    layers += [nn.Linear(width, 1)]
    return nn.Sequential(*layers)

X = torch.randn(256, 2)
y = torch.randn(256, 1)

print("gradient norm of the weight matrix at each depth (12 linear layers):")
print(f"{'':>9} {'layer 1':>9} {'layer 4':>9} {'layer 8':>9} {'layer 12':>9}   ratio 1/12")
for name, act in [("sigmoid", nn.Sigmoid), ("tanh", nn.Tanh), ("relu", nn.ReLU)]:
    torch.manual_seed(0)
    net = make_net(act)
    loss = ((net(X) - y) ** 2).mean()
    loss.backward()
    linears = [m for m in net if isinstance(m, nn.Linear)]
    g = [linears[i].weight.grad.norm().item() for i in (0, 3, 7, 11)]
    print(f"{name:>8}: {g[0]:9.2e} {g[1]:9.2e} {g[2]:9.2e} {g[3]:9.2e}   {g[0] / g[3]:.1e}")

## Universal approximation, informally — and why it is not enough


*Expected output starts with:* `width   2: max |error| = 1.1074`


In [ ]:
import numpy as np

np.random.seed(0)

# Target: a wiggly 1-D function on [0, 1]
x = np.linspace(0, 1, 500)
target = np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x)

def fit_relu_net(width):
    """One hidden layer of `width` ReLU units with random slopes and evenly
    spaced 'kinks'; only the output layer is fit (by least squares)."""
    kinks = np.linspace(0, 1, width, endpoint=False)          # where each ReLU bends
    H = np.maximum(0.0, x[:, None] - kinks[None, :])          # (500, width) hidden features
    H = np.hstack([H, np.ones((len(x), 1))])                  # bias column
    coef, *_ = np.linalg.lstsq(H, target)
    return H @ coef

for width in [2, 4, 8, 16, 32, 64]:
    pred = fit_relu_net(width)
    print(f"width {width:>3}: max |error| = {np.max(np.abs(pred - target)):.4f}")

## Going deeper


*Expected output starts with:* `depth  1 ( 2 ReLU units total):     2 linear pieces`


In [ ]:
import numpy as np

np.random.seed(0)

def tent(x):
    """The tent map on [0, 1] -- exactly two ReLU units: 2*relu(x) - 4*relu(x - 0.5)."""
    return 2 * np.maximum(0.0, x) - 4 * np.maximum(0.0, x - 0.5)

x = np.linspace(0.0, 1.0, 100001)
f = x.copy()
for depth in range(1, 11):
    f = tent(f)                                  # compose one more layer
    slopes = np.sign(np.diff(f))                 # slope sign on each tiny interval
    pieces = 1 + int(np.sum(slopes[1:] != slopes[:-1]))
    if depth in (1, 2, 3, 5, 8, 10):
        print(f"depth {depth:>2} ({2 * depth:>2} ReLU units total): {pieces:>5} linear pieces")

## Try it yourself

1. Modify the perceptron script to record the weight vector after each epoch on the XOR data. Does it cycle through a repeating sequence? How long is the cycle?
2. Hand-construct a 2-hidden-unit ReLU network that computes XOR on the four points `(±1, ±1)` (choose `W1`, `b1`, `w2`, `b2` yourself, no training), and verify it in NumPy.
3. In the width experiment, replace the ReLU features with `tanh((x - kinks) * 10)` features. How does max error scale with width now? What happens with the multiplier at 1 instead of 10?
4. Continue the width experiment to widths 128, 256, and 512 (increase the sample grid to 5000 points so the measurement stays honest). Does the factor-of-four-per-doubling trend continue? At what width does float64 arithmetic start to matter?
5. In the gradient-norm experiment, increase depth from 12 to 24 and rerun. Which activation's front-layer gradient degrades fastest? Then replace `nn.ReLU` with `nn.GELU` and compare.
6. Implement the delta rule (gradient descent on squared error of the pre-threshold sum) for the XOR points, and confirm that unlike the perceptron rule it converges — to weights near zero. Explain why "converges to the least-bad linear fit" and "solves the problem" are different claims.


---

Full discussion of everything above: [S02 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s02/).
